# 🔥 NEXinfra AI — 7-Class 7,000-Image YOLO Master Trainer
### Fine-Tune YOLO on 7 Municipal Defect Classes (1,000 Balanced Images / Class) on Colab GPU

**Supported 7 Canonical Civic Classes:**
1. `0: pothole_road_defect` (Road Damage / Pothole)
2. `1: water_drainage_burst` (Water / Drainage Burst)
3. `2: garbage_waste_overflow` (Solid Waste Overflow)
4. `3: electrical_hazard` (Electrical & Streetlight)
5. `4: structural_bridge_crack` (Structural Anomaly / Bridge Crack)
6. `5: tree_greenery_hazard` (Public Park & Greenery Hazard)
7. `6: fire_smoke_hazard` (Fire & Smoke Hazard — Critical Disaster Response)

---

In [ ]:
# Step 1: Install High-Performance YOLO & Data Augmentation Libraries
!pip install -q ultralytics onnx onnxslim onnxruntime albumentations opencv-python-headless pillow requests tqdm
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU!")

In [ ]:
# Step 2: Generate Balanced 7,000-Image Dataset (1,000 Samples Per Class)
import os
import cv2
import numpy as np
import yaml
from PIL import Image, ImageDraw, ImageFilter
import random
from tqdm import tqdm

dataset_root = '/content/nexinfra_7000_dataset'
os.makedirs(f'{dataset_root}/images/train', exist_ok=True)
os.makedirs(f'{dataset_root}/images/val', exist_ok=True)
os.makedirs(f'{dataset_root}/labels/train', exist_ok=True)
os.makedirs(f'{dataset_root}/labels/val', exist_ok=True)

CLASSES = {
    0: 'pothole_road_defect',
    1: 'water_drainage_burst',
    2: 'garbage_waste_overflow',
    3: 'electrical_hazard',
    4: 'structural_bridge_crack',
    5: 'tree_greenery_hazard',
    6: 'fire_smoke_hazard'
}

print('🚀 Synthesizing and Augmenting 7,000 Balanced Civic Defect Images (1,000/class)...')

def generate_defect_sample(class_id, sample_idx):
    # Canvas 640x640
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    
    # Base backgrounds
    if class_id in [0, 4]: # Asphalt road / Concrete wall
        base_gray = random.randint(45, 110)
        img[:, :] = (base_gray + np.random.randint(-15, 15, (640, 640, 3))).clip(0, 255)
    elif class_id == 1: # Wet roadway / water flood
        img[:, :] = [random.randint(60, 110), random.randint(70, 120), random.randint(90, 160)]
    elif class_id == 2: # Urban pavement garbage zone
        img[:, :] = [random.randint(70, 130), random.randint(65, 125), random.randint(60, 120)]
    elif class_id == 3: # Sky / pole / wire background
        img[:, :] = [random.randint(140, 210), random.randint(130, 190), random.randint(110, 170)]
    elif class_id == 5: # Park foliage background
        img[:, :] = [random.randint(30, 70), random.randint(70, 130), random.randint(30, 70)]
    elif class_id == 6: # Dark smoke / night industrial background
        img[:, :] = [random.randint(20, 50), random.randint(20, 50), random.randint(25, 55)]
    
    # Generate bounding box coordinates
    bw = random.randint(140, 360)
    bh = random.randint(120, 320)
    bx = random.randint(40, 640 - bw - 40)
    by = random.randint(40, 640 - bh - 40)
    
    # Draw class-specific visual signatures
    if class_id == 0: # Pothole: Dark crater + irregular cavity
        cv2.ellipse(img, (bx + bw//2, by + bh//2), (bw//2, bh//2), random.randint(0, 180), 0, 360, (20, 20, 22), -1)
        cv2.ellipse(img, (bx + bw//2, by + bh//2), (bw//3, bh//3), 0, 0, 360, (8, 8, 10), -1)
    elif class_id == 1: # Water Burst: Blue specular swirl + flood ripples
        for _ in range(12):
            cv2.circle(img, (bx + random.randint(0, bw), by + random.randint(0, bh)), random.randint(15, 60), (220, 180, 50), -1)
    elif class_id == 2: # Garbage Dump: Multi-color plastic bags & debris heap
        colors = [(0, 0, 220), (220, 200, 0), (0, 200, 0), (200, 200, 200), (20, 20, 20), (200, 0, 200)]
        for _ in range(25):
            c = random.choice(colors)
            cv2.rectangle(img, (bx + random.randint(0, bw-30), by + random.randint(0, bh-30)), 
                          (bx + random.randint(20, bw), by + random.randint(20, bh)), c, -1)
    elif class_id == 3: # Electrical Hazard: Dangling wire lines & orange spark flares
        cv2.line(img, (bx, by), (bx + bw, by + bh), (10, 10, 10), 3)
        cv2.circle(img, (bx + bw//2, by + bh//2), random.randint(15, 45), (0, 165, 255), -1)
    elif class_id == 4: # Structural Crack: High-contrast branched shear fissure
        pt1 = (bx + random.randint(0, bw//3), by + random.randint(0, bh//3))
        pt2 = (bx + bw - random.randint(0, bw//3), by + bh - random.randint(0, bh//3))
        cv2.line(img, pt1, pt2, (15, 15, 15), random.randint(3, 7))
    elif class_id == 5: # Fallen Tree / Greenery: Dark wood trunk + foliage limbs
        cv2.line(img, (bx, by + bh//2), (bx + bw, by + bh//2), (25, 45, 65), random.randint(15, 30))
        cv2.circle(img, (bx + bw//2, by + bh//2), random.randint(40, 90), (30, 140, 40), -1)
    elif class_id == 6: # Fire & Smoke: Intense red/orange/yellow flames + gray smoke
        # Smoke cloud
        for _ in range(15):
            cv2.circle(img, (bx + random.randint(0, bw), by + random.randint(0, bh//2)), random.randint(25, 75), (80, 80, 80), -1)
        # Flame core (Yellow, Orange, Red)
        for _ in range(20):
            cv2.circle(img, (bx + random.randint(bw//4, 3*bw//4), by + bh//3 + random.randint(0, bh//2)), random.randint(15, 55), (0, 69, 255), -1)
            cv2.circle(img, (bx + random.randint(bw//3, 2*bw//3), by + bh//2 + random.randint(0, bh//3)), random.randint(10, 35), (0, 215, 255), -1)

    # Gaussian noise & texture smoothing
    img = cv2.GaussianBlur(img, (3, 3), 0)
    
    # Normalized YOLO bbox format [class_id, cx, cy, w, h]
    norm_cx = (bx + bw / 2.0) / 640.0
    norm_cy = (by + bh / 2.0) / 640.0
    norm_w = bw / 640.0
    norm_h = bh / 640.0
    
    return img, f'{class_id} {norm_cx:.6f} {norm_cy:.6f} {norm_w:.6f} {norm_h:.6f}\n'

# Generate 1,000 images per class (850 train, 150 val) = 7,000 total images
total_images = 0
for class_id in range(7):
    class_name = CLASSES[class_id]
    print(f'   -> Generating 1,000 images for Class {class_id}: {class_name}')
    for i in tqdm(range(1000), desc=f'{class_name}'):
        split = 'train' if i < 850 else 'val'
        img, label = generate_defect_sample(class_id, i)
        
        filename = f'defect_c{class_id}_{split}_{i:04d}'
        cv2.imwrite(f'{dataset_root}/images/{split}/{filename}.jpg', img)
        with open(f'{dataset_root}/labels/{split}/{filename}.txt', 'w') as lf:
            lf.write(label)
        total_images += 1

# Write Dataset YAML
dataset_config = {
    'path': dataset_root,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 7,
    'names': CLASSES
}

with open('civic_7class_7000.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f'\n✅ Generated balanced dataset of {total_images} images across 7 classes!')
print('📁 Dataset Config: civic_7class_7000.yaml')

In [ ]:
# Step 3: Train YOLOv8 on GPU (7,000 Images, 50-75 Epochs)
from ultralytics import YOLO

print('🚀 Starting Ultralytics YOLOv8 Fine-Tuning on 7 Municipal Defect Classes...')

# Load base pretrained model
model = YOLO('yolov8n.pt')

results = model.train(
    data='civic_7class_7000.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    name='nexinfra_7class_run',
    save=True,
    plots=True,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    optimizer='AdamW'
)

print('🎉 Training completed successfully!')

In [ ]:
# Step 4: Export to Optimized ONNX Format
import glob
from ultralytics import YOLO

weights = glob.glob('runs/**/nexinfra_7class_run/weights/best.pt', recursive=True)
best_pt = weights[-1] if weights else 'runs/detect/nexinfra_7class_run/weights/best.pt'
print(f'✅ Found trained weights at: {best_pt}')

trained_model = YOLO(best_pt)
onnx_path = trained_model.export(
    format='onnx',
    imgsz=640,
    opset=17,
    simplify=True
)

print(f'📦 Exported 7-Class ONNX Model to: {onnx_path}')

In [ ]:
# Step 5: Automatically Download model.onnx & best.pt to your PC
from google.colab import files
import shutil
import os

onnx_files = glob.glob('runs/**/best.onnx', recursive=True)
if onnx_files:
    shutil.copy2(onnx_files[-1], 'model.onnx')
    print('📥 Downloading 7-class model.onnx...')
    files.download('model.onnx')
    
    if os.path.exists(best_pt):
        print('📥 Downloading best.pt...')
        files.download(best_pt)
else:
    print('❌ ONNX export file not found. Check training logs above.')